# Упражнение: Голова классификатора на CLS-векторах: warmup + linear decay

## Задание

Энкодер BERT заморожен, и для 60 отзывов уже посчитаны векторы токена `[CLS]` (сокращены до 8 чисел):
`input.csv` — `id, c0…c7, label` (0 — негатив, 1 — нейтрально, 2 — позитив).
Обучите поверх них линейную голову классификатора — то, что `AutoModelForSequenceClassification`
добавляет к BERT, — с расписанием learning rate «линейный разогрев + линейное затухание»
(как `get_linear_schedule_with_warmup` в Hugging Face).

- Модель: логиты z = X·W + b, W (8 × 3) и b (3) в начале **нулевые**; p = softmax(z).
- Потери: средняя кросс-энтропия по всем 60 объектам, L = −mean(log p_y).
- Обучение: полный батч, обычный градиентный спуск, всего T = 40 шагов, из них W = 4 шага разогрева,
  пиковый learning rate η = 0.5.
- Расписание для шага t = 1…T:
  - lrₜ = η · t / W при t ≤ W (разогрев от нуля);
  - lrₜ = η · (T − t) / (T − W) при t > W (линейно до нуля к последнему шагу).
- На шаге t: посчитать L при текущих весах (до обновления), затем градиент и обновление
  W ← W − lrₜ·∂L/∂W, b ← b − lrₜ·∂L/∂b.

## Подсказки

- Сначала выпишите таблицу lr: на шаге 4 он достигает пика 0.5, на шаге 40 равен нулю.
- При нулевых весах все вероятности равны 1/3, поэтому loss первого шага = ln 3 ≈ 1.0986 — проверьте себя.
- Градиент softmax + кросс-энтропии: G = (P − Y_onehot) / n, ∂L/∂W = Xᵀ·G, ∂L/∂b = сумма G по строкам.

## Ожидаемый формат выходного файла

CSV с колонками `step, lr, loss` — 40 строк (t = 1…40); `loss` — потери до обновления на этом шаге.
`lr` округлить до 6 знаков, `loss` — до 4.

## Проверка

Сохраните свой результат в `my_answer.csv` (в папке упражнения) и выполните из корня репозитория:

```bash
python tools/check_exercise.py exercises/bert_finetuning
```

Эталонное решение с пояснениями — `solution.py` (загляните в него только после своей попытки).
Верное решение портал отметит сам и откроет разбор (если он запущен через `python start.py`).

## Данные

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('input.csv')
print(df.shape)
df.head()

## Решение

In [ ]:
# Ваше решение. Итог запишите в DataFrame `result` — в формате, описанном в задании.

result = None

## Проверка

In [ ]:
# Проверка: сравнение result с эталоном (запускайте после своего решения).
# Сохраняет result в my_answer.csv, подсказывает вероятную причину ошибки, верное решение отмечает в портале.
import sys
from pathlib import Path

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'tools' / 'check_exercise.py').exists())
sys.path.insert(0, str(root / 'tools'))
from check_exercise import check_result

check_result(result)

---
Эталонное решение с пояснениями — в файле `solution.py`. Загляните в него после своей попытки.